In [1]:
%pip install laszip
%pip install -U laspy lazrs

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import numpy as np
import laspy

# ============================================================
# INPUT
# ============================================================
input_dir = Path("/home/b085164/PDM_Romain_Defferrard/ECCR")
output_dir = input_dir / "filtered_10xx_only"
output_dir.mkdir(parents=True, exist_ok=True)

recursive = True

# ============================================================
# DETECT AVAILABLE LAZ BACKEND
# ============================================================
available_backends = laspy.LazBackend.detect_available()
print("Available LAZ backends:", available_backends)

if not available_backends:
    raise RuntimeError(
        "No LAZ backend available. Install one with:\n"
        "  %pip install lazrs\n"
        "or\n"
        "  %pip install laszip"
    )

laz_backend = available_backends[0]
print("Using backend:", laz_backend)

# ============================================================
# HELPER
# Keep only flightlines 1000..1099
# ============================================================
def is_10xx_flightline(point_source_ids: np.ndarray) -> np.ndarray:
    point_source_ids = np.asarray(point_source_ids)
    return (point_source_ids >= 1000) & (point_source_ids < 1100)

# ============================================================
# FIND FILES
# ============================================================
pattern = "**/*.laz" if recursive else "*.laz"
laz_files = sorted(input_dir.glob(pattern))

print(f"{len(laz_files)} fichier(s) .laz trouvé(s)")

# ============================================================
# PROCESS
# ============================================================
for laz_path in laz_files:
    print("\n" + "=" * 80)
    print(f"Lecture : {laz_path}")

    try:
        las = laspy.read(laz_path, laz_backend=laz_backend)
    except Exception as e:
        print(f"  -> lecture impossible: {e}")
        continue

    dims = list(las.point_format.dimension_names)
    if "point_source_id" not in dims:
        print("  -> champ 'point_source_id' absent, fichier ignoré")
        continue

    psid = np.asarray(las["point_source_id"])
    vals, counts = np.unique(psid, return_counts=True)

    print("  Flightlines trouvées :")
    for v, c in zip(vals, counts):
        print(f"    {v}: {c} points")

    mask = is_10xx_flightline(psid)
    kept_ids = np.unique(psid[mask])

    n_total = len(psid)
    n_keep = int(mask.sum())

    print(f"  Flightlines gardées : {kept_ids.tolist()}")
    print(f"  Points gardés : {n_keep} / {n_total} ({100*n_keep/n_total:.2f} %)")

    if n_keep == 0:
        print("  -> aucun point conservé, fichier ignoré")
        continue

    las_filtered = las[mask]

    out_path = output_dir / f"{laz_path.stem}_10xx_only.laz"
    try:
        las_filtered.write(out_path, laz_backend=laz_backend)
        print(f"  -> écrit : {out_path}")
    except Exception as e:
        print(f"  -> écriture impossible: {e}")

print("\nTerminé.")

Available LAZ backends: (<LazBackend.LazrsParallel: 0>, <LazBackend.Lazrs: 1>, <LazBackend.Laszip: 2>)
Using backend: LazBackend.LazrsParallel
136 fichier(s) .laz trouvé(s)

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154700_10xx_only.laz
  Flightlines trouvées :
    1007: 8205156 points
    1008: 4733001 points
    1009: 11527112 points
    1012: 4885734 points
  Flightlines gardées : []
  Points gardés : 0 / 29351003 (0.00 %)
  -> aucun point conservé, fichier ignoré

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154800_10xx_only.laz
  Flightlines trouvées :
    1007: 3250925 points
    1008: 68877 points
    1009: 5486835 points
    1012: 10563147 points
    1015: 1457040 points
    1016: 7811077 points
  Flightlines gardées : []
  Points gardés : 0 / 28637901 (0.00 %)
  -> aucun point conservé, fichier ignoré

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154900_10xx_

KeyboardInterrupt: 

In [4]:
from pathlib import Path
import numpy as np
import laspy

# ============================================================
# INPUT
# ============================================================
input_dir  = Path("/home/b085164/PDM_Romain_Defferrard/ECCR")
output_dir = input_dir / "filtered_20xx_only_zone3"
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_FILES = [
    input_dir / "ECCR_MN95_NF02_2533200-1155100.laz",
    input_dir / "ECCR_MN95_NF02_2533200-1155200.laz",
]

# ============================================================
# DETECT AVAILABLE LAZ BACKEND
# ============================================================
available_backends = laspy.LazBackend.detect_available()
print("Available LAZ backends:", available_backends)
if not available_backends:
    raise RuntimeError(
        "No LAZ backend available. Install one with:\n"
        "  %pip install lazrs\n"
        "or\n"
        "  %pip install laszip"
    )
laz_backend = available_backends[0]
print("Using backend:", laz_backend)

# ============================================================
# HELPER — Keep only flightlines 2000..2099
# ============================================================
def is_20xx_flightline(point_source_ids: np.ndarray) -> np.ndarray:
    point_source_ids = np.asarray(point_source_ids)
    return (point_source_ids >= 2000) & (point_source_ids < 2100)

# ============================================================
# PROCESS
# ============================================================
for laz_path in TARGET_FILES:
    print("\n" + "=" * 80)
    print(f"Lecture : {laz_path}")

    if not laz_path.exists():
        print(f"  -> fichier introuvable: {laz_path}")
        continue

    try:
        las = laspy.read(laz_path, laz_backend=laz_backend)
    except Exception as e:
        print(f"  -> lecture impossible: {e}")
        continue

    dims = list(las.point_format.dimension_names)
    if "point_source_id" not in dims:
        print("  -> champ 'point_source_id' absent, fichier ignoré")
        continue

    psid = np.asarray(las["point_source_id"])
    vals, counts = np.unique(psid, return_counts=True)
    print("  Flightlines trouvées :")
    for v, c in zip(vals, counts):
        print(f"    {v}: {c} points")

    mask      = is_20xx_flightline(psid)
    kept_ids  = np.unique(psid[mask])
    n_total   = len(psid)
    n_keep    = int(mask.sum())
    print(f"  Flightlines gardées : {kept_ids.tolist()}")
    print(f"  Points gardés : {n_keep} / {n_total} ({100*n_keep/n_total:.2f} %)")

    if n_keep == 0:
        print("  -> aucun point conservé, fichier ignoré")
        continue

    las_filtered = las[mask]
    out_path = output_dir / f"{laz_path.stem}_20xx_only.laz"
    try:
        las_filtered.write(out_path, laz_backend=laz_backend)
        print(f"  -> écrit : {out_path}")
    except Exception as e:
        print(f"  -> écriture impossible: {e}")

print("\nTerminé.")

Available LAZ backends: (<LazBackend.LazrsParallel: 0>, <LazBackend.Lazrs: 1>, <LazBackend.Laszip: 2>)
Using backend: LazBackend.LazrsParallel

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ECCR_MN95_NF02_2533200-1155100.laz
  Flightlines trouvées :
    0: 154001966 points
    1003: 4621843 points
    1004: 6694964 points
    1010: 9859357 points
    1013: 6495058 points
    1019: 9720538 points
    1020: 4675397 points
    2008: 4508298 points
    2009: 13144530 points
    2010: 18323062 points
    2011: 2181226 points
    2014: 2936709 points
    2015: 16347471 points
    2016: 9239230 points
    3008: 3130719 points
    3009: 10715855 points
    3010: 13836715 points
    3011: 2013253 points
    3014: 2309101 points
    3015: 12303697 points
    3016: 9857633 points
  Flightlines gardées : [2008, 2009, 2010, 2011, 2014, 2015, 2016]
  Points gardés : 66680526 / 316916622 (21.04 %)
  -> écrit : /home/b085164/PDM_Romain_Defferrard/ECCR/filtered_20xx_only_zone3/ECCR_MN95_NF02_25332

In [ ]:
import folium
from folium.plugins import Draw
from IPython.display import display
from pathlib import Path

geojson_out = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/single_roi.geojson")

center_lat = 46.53
center_lon = 6.64

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    control_scale=True
)

Draw(
    export=True,
    filename=geojson_out.name,
    draw_options={
        "polyline": False,
        "rectangle": True,
        "polygon": True,
        "circle": False,
        "marker": False,
        "circlemarker": False,
    },
    edit_options={"edit": True}
).add_to(m)

display(m)

In [ ]:
import folium
from folium.plugins import Draw
from pathlib import Path

geojson_out = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/single_roi.geojson")
html_out = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/map_zones_interet.html")

m = folium.Map(location=[46.53, 6.64], zoom_start=14, control_scale=True)

Draw(
    export=True,
    filename=geojson_out.name,
    draw_options={
        "polyline": False,
        "rectangle": True,
        "polygon": True,
        "circle": False,
        "marker": False,
        "circlemarker": False,
    },
    edit_options={"edit": True}
).add_to(m)

m.save(html_out)
print(f"Carte sauvée dans : {html_out}")

Carte sauvée dans : /home/b085164/PDM_Romain_Defferrard/ECCR/map_zones_interet.html


In [5]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import laspy
from shapely.ops import unary_union
from shapely import contains_xy

# ============================================================
# INPUTS
# ============================================================
input_dir = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only")
zones_path = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/zone_3.geojson")
output_dir = input_dir / "clipped_roi_3"
output_dir.mkdir(parents=True, exist_ok=True)

recursive = True
lidar_crs = "EPSG:2056"   # CRS des .laz

# ============================================================
# DETECT LAZ BACKEND
# ============================================================
available_backends = laspy.LazBackend.detect_available()
print("Available LAZ backends:", available_backends)

if not available_backends:
    raise RuntimeError(
        "No LAZ backend available. Install one with:\n"
        "  %pip install lazrs\n"
        "or\n"
        "  %pip install laszip"
    )

laz_backend = available_backends[0]
print("Using backend:", laz_backend)

# ============================================================
# LOAD ROI POLYGONS
# Folium export = EPSG:4326 (WGS84)
# -> reprojection vers le CRS LiDAR
# ============================================================
gdf = gpd.read_file(zones_path)

if gdf.empty:
    raise ValueError("Le GeoJSON ne contient aucune géométrie.")

print("CRS GeoJSON lu :", gdf.crs)

if gdf.crs is None:
    print("Aucun CRS détecté dans le GeoJSON -> on suppose EPSG:4326")
    gdf = gdf.set_crs("EPSG:4326")

gdf = gdf.to_crs(lidar_crs)

print("CRS après reprojection :", gdf.crs)
print(f"{len(gdf)} polygone(s) chargé(s)")
print("Bounds ROI reprojetés :", gdf.total_bounds)

# Fusion en une géométrie unique pour accélérer le test
roi_union = unary_union(gdf.geometry)

# ============================================================
# FIND LAZ FILES
# ============================================================
pattern = "**/*.laz" if recursive else "*.laz"
laz_files = sorted(input_dir.glob(pattern))

print(f"{len(laz_files)} fichier(s) .laz trouvé(s)")

# ============================================================
# PROCESS
# ============================================================
for laz_path in laz_files:
    if output_dir in laz_path.parents:
        continue

    print("\n" + "=" * 80)
    print("Lecture :", laz_path)

    try:
        las = laspy.read(laz_path, laz_backend=laz_backend)
    except Exception as e:
        print("  -> lecture impossible:", e)
        continue

    # Coordonnées LiDAR
    x = las.x
    y = las.y

    # Préfiltre bbox
    minx, miny, maxx, maxy = roi_union.bounds
    bbox_mask = (x >= minx) & (x <= maxx) & (y >= miny) & (y <= maxy)

    n_bbox = int(bbox_mask.sum())
    if n_bbox == 0:
        print("  -> aucun point dans la bbox ROI, fichier ignoré")
        continue

    # Test exact dans le polygone
    idx_bbox = np.where(bbox_mask)[0]
    inside_bbox = contains_xy(roi_union, x[bbox_mask], y[bbox_mask])

    mask = np.zeros(len(x), dtype=bool)
    mask[idx_bbox] = inside_bbox

    n_keep = int(mask.sum())
    n_total = len(x)

    print(f"  Points gardés : {n_keep} / {n_total} ({100*n_keep/n_total:.2f} %)")

    if n_keep == 0:
        print("  -> aucun point dans les polygones, fichier ignoré")
        continue

    las_clipped = las[mask]

    out_path = output_dir / f"{laz_path.stem}_roi.laz"
    try:
        las_clipped.write(out_path, laz_backend=laz_backend)
        print("  -> écrit :", out_path)
    except Exception as e:
        print("  -> écriture impossible:", e)

print("\nTerminé.")

Available LAZ backends: (<LazBackend.LazrsParallel: 0>, <LazBackend.Lazrs: 1>, <LazBackend.Laszip: 2>)
Using backend: LazBackend.LazrsParallel
CRS GeoJSON lu : EPSG:4326
CRS après reprojection : EPSG:2056
1 polygone(s) chargé(s)
Bounds ROI reprojetés : [2533163.69798862 1154966.41733544 2533311.1747593  1155283.78096532]
77 fichier(s) .laz trouvé(s)

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154700_10xx_only.laz
  -> aucun point dans la bbox ROI, fichier ignoré

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154800_10xx_only.laz
  -> aucun point dans la bbox ROI, fichier ignoré

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532400-1154900_10xx_only.laz
  -> aucun point dans la bbox ROI, fichier ignoré

Lecture : /home/b085164/PDM_Romain_Defferrard/ECCR/ALS_only/ECCR_MN95_NF02_2532500-1154700_10xx_only.laz
  -> aucun point dans la bbox ROI, fichier ignoré

Lecture : /home/b085164